# Peer-reliability recovery — current-API stress test (GPU sweep)

25 people with latent reliability `r_i ~ Beta(8, 2)`; each rates 6 random
others with noise whose scale depends on the **rater's own** reliability.
Ground truth is known by construction, so scoring needs no reference
posterior: **RMSE of the posterior mean** vs the eigenvector-style
iterative baseline and the prior-shrunk mean, plus **80%-interval
coverage** (want 0.80 — the baselines have no intervals at all).

This is deliberately a test of the **current `DistributionBuilder` API** —
no solver changes. Two encodings per replication:

* `fixed` — every eval is an `ExpectationEstimate` with one global sd
  (misspecified: rater identity discarded; should land near the shrunk
  mean, measures graceful degradation under ~150 inconsistent constraints).
* `eqn` — the heteroscedastic model expressed in the *existing* equation
  language via a variance-stabilised residual
  (`r_j = r_j - (r_j - s)*SIG0/sigma(r_i) ~ N(0, SIG0)`); the arm that can
  beat the baselines.

All logic lives in `benchmarks/reliability_experiment.py`; this notebook is
a thin shell (clone-or-pull -> run -> summarize). Iterate by editing the
payload, pushing to `main`, and re-running the setup cell.

**Runtime -> Change runtime type -> GPU** (T4 is fine), then Run all.

In [ ]:
# --- Setup: clone-or-pull the repo, install deps Colab lacks, keep Colab's jax ---
import os, sys, subprocess

REPO = "/content/calibrated_response"
BRANCH = "main"   # must carry the reliability-benchmark commit
URL = "https://github.com/amdson/calibrated_response.git"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"],
                   check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

%pip -q install optax jaxopt pydantic

import jax
jax.config.update("jax_compilation_cache_dir", f"{REPO}/.jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("jax backend:", jax.default_backend(), jax.devices())
assert jax.default_backend() != "cpu", "No GPU — switch the runtime type first"

# import-guard: confirm the pulled branch carries the payload
from benchmarks.reliability_experiment import (run_sweep, summarize,
                                               default_configs, quick_configs,
                                               flip_configs)
print(f"payload OK — {len(default_configs())} runs in the default grid")

In [ ]:
# --- Sweep knobs -----------------------------------------------------------
# MODE = "quick": pop=10, 2 seeds, short fits — validates the notebook (~min).
# MODE = "full":  pop=25, n_evals=6, fixed+eqn x 5 seeds at 3000 steps.
# MODE = "flip":  the HARD variant — unreliable raters invert reviews
#                 (p_flip = (1-r)^2) before noising: bimodal likelihood,
#                 "target is bad" vs "rater is a flipper" are competing
#                 explanations.  Persons 0..2 are guaranteed duds
#                 (r ~ Beta(2,6), n_bad=3) so near-inverters exist on every
#                 seed — the solver's priors still say E[r]=0.8 for all.
#                 The eig baseline can down-weight but never invert a
#                 rater; the eqn arm gets the true mixture moment-matched.
#                 fixed stays flip-blind.
MODE = "full"

CONFIGS = (quick_configs() if MODE == "quick"
           else flip_configs() if MODE == "flip"
           else default_configs())
OUT = "results/reliability.jsonl"
print(f"{MODE}: {len(CONFIGS)} runs -> {OUT}")

In [ ]:
# --- Run the sweep (resumable: rows append to the OUT jsonl) ----------------
# fits: _key(row) -> inspection dict {builder, r, evals, samples, mean,
# q10, q90, b_mean, b_eig} for every run executed THIS call (resumed rows
# have no live sampler — delete the jsonl row to re-fit one).
rows, fits = run_sweep(CONFIGS, out_path=OUT)

In [ ]:
# --- Summarize ---------------------------------------------------------------
# rmse_fit vs rmse_mean / rmse_eig: the eqn arm competes for the mean->eig
# headroom; cover80 wants 0.80 (the calibration claim baselines can't make).
summarize(OUT)

In [ ]:
# --- Inspect one fitted sampler ---------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

key = sorted(fits)[-1]              # (pop, n_evals, mode, steps, flip, seed)
fit = fits[key]
print("inspecting", key)
b, r = fit["builder"], fit["r"]

# per-person: truth vs posterior mean + 80% interval, vs the baselines
order = np.argsort(r)
plt.figure(figsize=(9, 4))
plt.errorbar(np.arange(len(r)), fit["mean"][order],
             yerr=[fit["mean"][order] - fit["q10"][order],
                   fit["q90"][order] - fit["mean"][order]],
             fmt="o", ms=4, lw=1, capsize=2, label="fit (80% interval)")
plt.plot(r[order], "k_", ms=12, label="truth")
plt.plot(fit["b_eig"][order], "x", ms=5, alpha=0.7, label="eig baseline")
plt.xlabel("person (sorted by true r)"); plt.ylabel("reliability")
plt.legend(); plt.title(str(key)); plt.tight_layout(); plt.show()

# constraint satisfaction: worst-fitted estimates first
rep = sorted(b.constraint_report(), key=lambda c: -abs(c["error_rel"]))
for c in rep[:12]:
    print(f"{c['id']:>14}  target={c['target']:.3f} "
          f"fitted={c['fitted']:.3f}  err_rel={c['error_rel']:+.3f}")

s = b.sample_dict(50_000, seed=7)           # dict name -> (N,) samples

In [ ]:
# --- Pairwise (corner) plot of a variable subset ------------------------------
# The joint structure the summary metrics can't show: does a dud's marginal
# come out bimodal (good-vs-bad hypothesis kept alive)?  Do hypothesis
# correlations appear ("if rater 0 is a flipper, target 5 is actually
# good")?  Dashed lines mark the TRUE reliabilities.
from calibrated_response.maxent_sampler import plot_pairwise
import numpy as np

fit = fits[key]                             # key from the inspection cell
b, r = fit["builder"], fit["r"]

PERSONS = None      # e.g. [0, 1, 2, 8, 17] — indices into the population
if PERSONS is None:
    # default: the duds (lowest true r) + the person the fit misses worst
    # + the best-rated person, capped at 6 panels
    by_truth = list(np.argsort(r))
    worst_fit = int(np.argmax(np.abs(fit["mean"] - r)))
    PERSONS = list(dict.fromkeys(by_truth[:3] + [worst_fit] + by_truth[-2:]))[:6]

names = [f"p{i:02d}" for i in PERSONS]
sites = [b.var_name_to_idx[f"rel_{i:02d}"] for i in PERSONS]
plot_pairwise(b.model, b.params, sites=sites, names=names,
              n_samples=30_000, seed=11, bins=50,
              threshold={n: float(r[i]) for n, i in zip(names, PERSONS)});

In [ ]:
# --- Download results (unzip into results/ locally to merge) ---
import shutil
shutil.make_archive("/content/reliability_results", "zip", "results")
try:
    from google.colab import files
    files.download("/content/reliability_results.zip")
except ImportError:
    print("not on Colab — results in results/")